In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from torchvision.models import VGG16_Weights

In [12]:
from priorbox import AnchorBoxes
import yaml
from utils import normalised_gt_coords,corner_to_center,iou,center_to_corner

with open('config/priorbox.yaml', 'r') as file:
        config = yaml.safe_load(file)

boxes=AnchorBoxes(config)
anchors=boxes.forward()
gt_boxes = [
    normalised_gt_coords(75, 80, 60, 90,300,300),
    normalised_gt_coords(220, 70, 50, 40,300,300),
    normalised_gt_coords(150, 170, 140, 110,300,300),
    normalised_gt_coords(260, 240, 70, 120,300,300),
    normalised_gt_coords(45, 230, 80, 60,300,300)
]
gtboxes=torch.tensor(gt_boxes, dtype=torch.float32)



In [36]:
gtboxes

def normalised_gt_coords(box,H,W):
    
    """
    normalise gt coords so that they are in [0,1]
    """

    return torch.cat((box[:,0]/W,box[:,1]/H,box[:,2]/W,box[:,3]/H))

In [25]:

def center_to_corner(cx, cy, w, h):
    x1 = cx - w / 2
    y1 = cy - h / 2
    x2 = cx + w / 2
    y2 = cy + h / 2
    return x1, y1, x2, y2

center_to_corner(*el.tolist())

(0.0, 0.0, 0.14354194700717926, 0.14354194700717926)

In [26]:
anchors

tensor([[0.0718, 0.0718, 0.1435, 0.1435],
        [0.0849, 0.0718, 0.1699, 0.1435],
        [0.0981, 0.0718, 0.1962, 0.1435],
        ...,
        [0.5000, 0.5000, 0.9000, 0.9000],
        [0.5000, 0.5000, 1.0000, 0.6364],
        [0.5000, 0.5000, 0.6364, 1.0000]])

In [27]:
center_to_corner(*el.tolist())

(0.0, 0.0, 0.14354194700717926, 0.14354194700717926)

In [29]:
def iou(x1A,y1A,x2A,y2A,x1B,y1B,x2B,y2B):
    a=min(x2A,x2B)
    b=max(x1B,x1A)
    d=max(y1A,y1B)
    c=min(y2A,y2B)

    h=max(0,a-b)
    w=max(0,c-d)
    intersection=h*w

    A_area=(x2A-x1A)*(y2A-y1A)
    B_area=(x2B-x1B)*(y2B-y1B)

    union=A_area+B_area-intersection

    return intersection/union
    
maxx=0
for i, el in enumerate(anchors):
    for j,jel in enumerate(gtboxes):
        val= iou(*center_to_corner(*el.tolist()),*center_to_corner(*jel.tolist()))
        maxx=max(maxx,val)
        if val>0.5:
            print(i,j,val)
print(maxx)

312 0 0.589673411787272
313 0 0.589673411787272
314 0 0.589673411787272
349 0 0.5677085887182899
350 0 0.6876217164057973
351 0 0.6876217164057973
352 0 0.6876217164057973
353 0 0.5677085141085983
387 0 0.5677085887182899
388 0 0.6876217164057973
389 0 0.6876217164057973
390 0 0.6876217164057973
391 0 0.5677085141085983
425 0 0.513827411792137
426 0 0.6178740722670548
427 0 0.6178740722670548
428 0 0.6178740722670548
429 0 0.5138273465845576
464 0 0.5169592647912782
465 0 0.5169592647912782
466 0 0.5169592647912782
993 4 0.5223752183556676
1021 3 0.5018820903229134
1029 4 0.5299740608388539
1030 4 0.5822879061016591
1031 4 0.6644123411789866
1032 4 0.5994887339663555
1033 4 0.5018510109020619
1058 3 0.5809128306601716
1059 3 0.5943380092631906
1060 3 0.5279004954745408
1066 4 0.5234370075493341
1067 4 0.5864986026595028
1068 4 0.646750583400655
1069 4 0.7422314555772619
1070 4 0.6666579447765375
1071 4 0.5542894111999332
1095 3 0.5113704850892319
1096 3 0.6055073676594687
1097 3 0.6197

In [ ]:
gtboxes

tensor([[0.2500, 0.2667, 0.2000, 0.3000],
        [0.7333, 0.2333, 0.1667, 0.1333],
        [0.5000, 0.5667, 0.4667, 0.3667],
        [0.8667, 0.8000, 0.2333, 0.4000],
        [0.1500, 0.7667, 0.2667, 0.2000]])

In [251]:
anchors.shape

torch.Size([8732, 4])

In [ ]:
def center_to_corner(box):
    """from cx,cy,w,h to x1,y1,x2,y2"""

    return torch.cat((box[:,:2]-box[:,2:4]/2,box[:,:2]+box[:,2:4]/2,),1)
anch=center_to_corner(anchors)
gt=center_to_corner(gtboxes)

In [139]:

def iou(anchors,gtboxes):

    """
    anchors , gtboxes are in center to corner version

    """
    nb_anchors=anchors.shape[0]
    n_gt=gtboxes.shape[0]

    a=torch.min(anchors[:,2].unsqueeze(1),gtboxes[:,2].unsqueeze(0))
    b=torch.max(anchors[:,0].unsqueeze(1),gtboxes[:,0].unsqueeze(0))
    d=torch.max(anchors[:,1].unsqueeze(1),gtboxes[:,1].unsqueeze(0))
    c=torch.min(anchors[:,3].unsqueeze(1),gtboxes[:,3].unsqueeze(0))

    h=torch.clamp((a-b),min=0)
    w=torch.clamp((c-d),min=0)
    intersection=h*w

    A_area=(anchors[:,2].unsqueeze(1)-anchors[:,0].unsqueeze(1))*(anchors[:,3].unsqueeze(1)-anchors[:,1].unsqueeze(1))
    B_area=(gtboxes[:,2].unsqueeze(1)-gtboxes[:,0].unsqueeze(1))*(gtboxes[:,3].unsqueeze(1)-gtboxes[:,1].unsqueeze(1))
    union=A_area+B_area.unsqueeze(0)[:,:,0].expand(nb_anchors,n_gt)-intersection

    return intersection/union

In [140]:
intersection=iou(anch,gt)
intersection

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0064, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0143, 0.0000, 0.0000, 0.0000, 0.0000],
        ...,
        [0.0741, 0.0274, 0.2112, 0.0840, 0.0571],
        [0.0723, 0.0308, 0.2689, 0.0750, 0.0622],
        [0.0781, 0.0349, 0.2689, 0.0388, 0.0303]])

In [62]:
intersection.shape

torch.Size([8732, 5])

In [137]:
# for each gt best overlap ;
for_gt_id=intersection.argmax(dim=0)
# for each anchort best overlap 
for_anchor_iou,for_anchor_idx, = intersection.max(dim=1,keepdim=True)

for i in range(gt.shape[0]):
    for_anchor_idx[for_gt_id[i]]=i
for_anchor_idx



tensor([[0],
        [0],
        [0],
        ...,
        [2],
        [2],
        [2]])

In [138]:
gt

tensor([[0.1500, 0.1167, 0.3500, 0.4167],
        [0.6500, 0.1667, 0.8167, 0.3000],
        [0.2667, 0.3833, 0.7333, 0.7500],
        [0.7500, 0.6000, 0.9833, 1.0000],
        [0.0167, 0.6667, 0.2833, 0.8667]])

In [132]:
for_anchor_labels>0

tensor([[False],
        [False],
        [False],
        ...,
        [False],
        [False],
        [False]])

In [133]:
for_anchor_labels[312]

tensor([0])

In [119]:
(for_anchor_labels+1).max()

tensor(5)

In [116]:
for_gt_id[0]

tensor(389)

In [87]:
import torch
# Create a 2D tensor
x = torch.tensor([[1, 2, 3], [4, 5, 6], [7, 8, 9]], dtype=torch.float)
# Specify indices to fill
index = torch.tensor([0, 2])
# Fill the specified indices along dimension 1 with -1
x.index_fill_(1, index, -1)
print(x)

tensor([[-1.,  2., -1.],
        [-1.,  5., -1.],
        [-1.,  8., -1.]])


In [10]:
vgg = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features
vgg[16] = nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True)

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /home/onyxia/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100.0%


In [11]:
base=nn.ModuleList(vgg[:30])#until 5_3 layer

In [12]:
base
convs={}
i=1
convid=1
print("----","conv",0,"----")

for id, el in enumerate(base):

    print(f"({id})",el )
    if isinstance(el,nn.Conv2d):
        
        convs[id]=f"{i}_{convid}"
        convid=convid+1
    if isinstance(el,nn.MaxPool2d):
        convid=1
        print("----","conv",i,"----")
        i=i+1
    

    

    
    

---- conv 0 ----
(0) Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
(1) ReLU(inplace=True)
(2) Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
(3) ReLU(inplace=True)
(4) MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
---- conv 1 ----
(5) Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
(6) ReLU(inplace=True)
(7) Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
(8) ReLU(inplace=True)
(9) MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
---- conv 2 ----
(10) Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
(11) ReLU(inplace=True)
(12) Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
(13) ReLU(inplace=True)
(14) Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
(15) ReLU(inplace=True)
(16) MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=True)
---- conv 3 ----
(17) Conv2d(256, 512, kernel_s

In [13]:
convs

{0: '1_1',
 2: '1_2',
 5: '2_1',
 7: '2_2',
 10: '3_1',
 12: '3_2',
 14: '3_3',
 17: '4_1',
 19: '4_2',
 21: '4_3',
 24: '5_1',
 26: '5_2',
 28: '5_3'}

In [14]:
from l2norm import L2norm
class SSD(nn.Module):
    def __init__(self,base,nb_classes):
        super().__init__()
        self.features=base
        self.nb_classes=nb_classes

        self.l2norm=L2norm(512,20)

        self.extras=nn.ModuleList([
            #conv6 and conv7
            nn.Sequential(
                nn.Conv2d(in_channels=512,out_channels=1024,kernel_size=3,padding=6,dilation=6),
                nn.ReLU(inplace=True),
                nn.Conv2d(in_channels=1024,out_channels=1024,kernel_size=1),
                nn.ReLU(inplace=True),    
            ),

            #conv8_2
            nn.Sequential(
                nn.Conv2d(in_channels=1024,out_channels=256,kernel_size=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(in_channels=256,out_channels=512,kernel_size=3,padding=1,stride=2),
                nn.ReLU(inplace=True),    
            ),

            #conv9_2
            nn.Sequential(
                nn.Conv2d(in_channels=512,out_channels=128,kernel_size=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(in_channels=128,out_channels=256,kernel_size=3,padding=1,stride=2),
                nn.ReLU(inplace=True),    
            ),

            #conv10_2
            nn.Sequential(
                nn.Conv2d(in_channels=256,out_channels=128,kernel_size=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(in_channels=128,out_channels=256,kernel_size=3),
                nn.ReLU(inplace=True),    
            ),

            #conv11_2
            nn.Sequential(
                nn.Conv2d(in_channels=256,out_channels=128,kernel_size=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(in_channels=128,out_channels=256,kernel_size=3),
                nn.ReLU(inplace=True),    
            )
        ])

        #define kernels for classification to output Feature Map of sieze H,W,ki*nb_classes for all H,W, ki in {4,6} for all i =1 ... |ssd feature maps | , ki is number of anchors for each location of H,W, image 
        self.classification_convolutions=nn.ModuleList([

            nn.Conv2d(512,4*nb_classes,kernel_size=3,padding=1),
            nn.Conv2d(1024,6*nb_classes,kernel_size=3,padding=1),
            nn.Conv2d(512,6*nb_classes,kernel_size=3,padding=1),
            nn.Conv2d(256,6*nb_classes,kernel_size=3,padding=1),
            nn.Conv2d(256,4*nb_classes,kernel_size=3,padding=1),
            nn.Conv2d(256,4*nb_classes,kernel_size=3,padding=1),
        ])

        #same but using 4 coordinates for each anchor 
        self.regression_convolutions=nn.ModuleList([

            nn.Conv2d(512,4*4,kernel_size=3,padding=1),
            nn.Conv2d(1024,6*4,kernel_size=3,padding=1),
            nn.Conv2d(512,6*4,kernel_size=3,padding=1),
            nn.Conv2d(256,6*4,kernel_size=3,padding=1),
            nn.Conv2d(256,4*4,kernel_size=3,padding=1),
            nn.Conv2d(256,4*4,kernel_size=3,padding=1),

        ])
    def forward(self, X):
        layers_for_prediction = []
      
        #base model 
        for idx in range(len(self.features)):

            X=self.features[idx](X)
            
            if idx in convs and convs[idx]=="4_3":
                # print(self.features[idx])
                # print(X.shape)

                X=self.l2norm(X)
                layers_for_prediction.append(X)
          
                
        for idx in range(len(self.extras)):
            X=self.extras[idx](X)
            
            layers_for_prediction.append(X)


        classifications=[]    
        for layer_for_predictions, classification_convolution in zip( layers_for_prediction, self.classification_convolutions):

            x=classification_convolution(layer_for_predictions)
            #then we want to get for all i,j in H*H and all k in 1....K -> p1.....pC probabilities of C classes
            """

            mathematically : 

            anchors=6
            total=6*21
            classes=21
            N=10
            H=19
            x=torch.randn((N,H,H,total))
            x.view(N,H,H,anchors,int(total/anchors)).shape

            x.view(N,H,H,anchors,int(total/anchors)).view(N,H*H*anchors,classes).shape

            However, this iplementation is slower as need to track nb_anchors and do manual calculations 
            which will slow down the process, this is why we do more standard code (this comment is for self learning purpose)
            """
            classifications.append(x.permute(0,2,3,1).contiguous())
        
        regressions=[]  
        for layer_for_predictions, regression_convolution in zip( layers_for_prediction, self.regression_convolutions):
            x=regression_convolution(layer_for_predictions)
            regressions.append(x.permute(0,2,3,1).contiguous())

        #this efficient code was taken from degroot/ssd.pytorch github and is equivalent to my code in comment
        
        loc = torch.cat([o.view(o.size(0), -1) for o in regressions], 1)
        conf = torch.cat([o.view(o.size(0), -1) for o in classifications], 1)


        locs = loc.view(loc.size(0), -1, 4)#so we get for every image 2D matrix for classification and regression : 8732 anchor boxes and nb coords/classes
        #8732 anchor boxes are sum of all anchor boxes across all ft map k =  of sum over k (Hk*Hk*ak) 
        #for standard ssd300 it is 38*38*4+19*19*6+100*6+25*6+9*4+4
        confs = conf.view(conf.size(0), -1, self.nb_classes)

        return locs,confs,layers_for_prediction



In [15]:
X=torch.randn((10,3,300,300))

model=SSD(base,21)
classifications,regressions,layers_for_prediction=model.forward(X)

In [16]:
classifications.shape

torch.Size([10, 8732, 4])

In [17]:
regressions.shape



torch.Size([10, 8732, 21])

In [30]:
for el  in layers_for_prediction:
    print(el.shape[2:])



torch.Size([38, 38])
torch.Size([19, 19])
torch.Size([10, 10])
torch.Size([5, 5])
torch.Size([3, 3])
torch.Size([1, 1])
